<a href="https://colab.research.google.com/github/fares3010/classification/blob/main/Mice_Behavior_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Social Action Recognition in Mice
================================================================
Memory-optimized 2-pass architecture:

  Pass 1: Streaming snippet collection with reservoir sampling + incremental PCA via partial_fit

  Cluster: MiniBatchKMeans on capped reservoir (never >MAX_RESERVOIR rows)

  Pass 2: Per-video feature extraction → immediate downsample → flush to disk

  Train: Stream chunks from disk, cap total rows per action

  Infer: One video at a time, no accumulation

**Key memory savings:**

  • Reservoir cap on snippet arrays

  • Per-video flush to .npy in Pass 2

  • Downsampling happens per-video

  • NumPy arrays instead of DataFrames for intermediate features

  • Aggressive del + gc.collect() at every stage

In [1]:
import os, sys, gc, ast, warnings, traceback, time, tempfile, shutil
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
from scipy.ndimage import uniform_filter1d
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import MiniBatchKMeans
import joblib

warnings.filterwarnings("ignore")

#  CONFIG

UNSUP_MAX_VIDS   = 1200
SNIPPET_SAMPLE   = 80
MAX_RESERVOIR_S  = 150_000
MAX_RESERVOIR_P  = 150_000
N_CLUST_S        = 50
N_CLUST_P        = 80

# Pass 2 / Training caps
MAX_ROWS_PER_ACT = 300_000
NEG_RATIO        = 5.0
PAD_FRAMES       = 150
MAX_FRAMES_VID   = 50_000

# Temp dir for flushing intermediate chunks
_TMPDIR = None

def get_tmpdir():
    global _TMPDIR
    if _TMPDIR is None:
        base = Path("/content/drive/MyDrive/MABe-mouse-behavior-detection") if Path("/content/drive/MyDrive/MABe-mouse-behavior-detection").exists() else Path("/content")
        _TMPDIR = tempfile.mkdtemp(prefix="mabe_chunks_", dir=base)
    return Path(_TMPDIR)

def cleanup_tmpdir():
    global _TMPDIR
    if _TMPDIR and Path(_TMPDIR).exists():
        shutil.rmtree(_TMPDIR, ignore_errors=True)
        _TMPDIR = None

#  PATHS
def find_root():
    cands  = [
        Path("/content/drive/MyDrive/MABe-mouse-behavior-detection"),
        Path("/content/drive/MyDrive/MABe-mouse-behavior-detection (1)"),
        Path("./data"), Path("."),
    ]
    for r in cands:
        if r.exists() and (r / "train.csv").exists():
            return r
    sys.exit("[FATAL] train.csv not found")


if Path("/content").exists():        OUT = Path("/content")
else:                                  OUT = Path(".")
MDIR = OUT / "models"; MDIR.mkdir(parents=True, exist_ok=True)

#  HELPERS

def _sc(b, fps, ref=30.0):
    return max(1, int(round(b * fps / ref)))


def parse_beh(s):
    try:
        items = ast.literal_eval(s)
    except Exception:
        return []
    return [
        (p[0].strip(), p[1].strip(), p[2].strip())
        for item in items
        for p in [item.split(",")]
        if len(p) == 3
    ]


def sanitize_arr(X):
    """In-place sanitize a numpy float32 array."""
    np.nan_to_num(X, copy=False, nan=0.0, posinf=0.0, neginf=0.0)
    return X


def sanitize(X):
    """Sanitize a DataFrame."""
    X = X.replace([np.inf, -np.inf], np.nan)
    X = X.interpolate(axis=0, limit_direction="both")
    X = X.fillna(X.median()).fillna(0.0)
    return X


def _find_pq(base, lab, vid):
    p = base / lab / f"{vid}.parquet"
    if p.exists():
        return p
    p2 = base / f"{vid}.parquet"
    return p2 if p2.exists() else None

# RESERVOIR SAMPLER  (fixed-size, streaming)



In [2]:
class Reservoir:
    """Reservoir sampling into a pre-allocated numpy array."""

    def __init__(self, max_rows, ncols, seed=42):
        self.max_rows = max_rows
        self.buf = np.empty((max_rows, ncols), dtype="float32")
        self.n = 0          # rows seen so far
        self.filled = 0     # rows actually stored
        self.rng = np.random.RandomState(seed)

    def add(self, block):
        """Add a block of rows, keeping at most max_rows via reservoir sampling."""
        for i in range(len(block)):
            if self.filled < self.max_rows:
                self.buf[self.filled] = block[i]
                self.filled += 1
            else:
                j = self.rng.randint(0, self.n + 1)
                if j < self.max_rows:
                    self.buf[j] = block[i]
            self.n += 1

    def get(self):
        return self.buf[:self.filled].copy()

# DATA LOADING — IDs normalized at load

In [3]:
def load_track(vid, lab, tdir, ppc):
    fp = _find_pq(tdir, lab, vid)
    if fp is None:
        return None, {}
    df = pd.read_parquet(fp)
    if not {"video_frame", "mouse_id", "bodypart", "x", "y"}.issubset(df.columns):
        return None, {}
    unique = sorted(df["mouse_id"].unique())
    idmap = {old: f"mouse{i+1}" for i, old in enumerate(unique)}
    df["mouse_id"] = df["mouse_id"].map(idmap)
    pvid = df.pivot_table(
        index="video_frame", columns=["mouse_id", "bodypart"],
        values=["x", "y"], aggfunc="first",
    )
    pvid = pvid.reorder_levels([1, 2, 0], axis=1).sort_index(axis=1)
    pvid = (pvid / max(ppc, 0.1)).astype("float32")
    pvid = pvid.interpolate(limit_direction="both").fillna(method="bfill").fillna(method="ffill")
    nn = pvid.columns[pvid.isna().all()]
    if len(nn):
        pvid = pvid.drop(columns=nn)
    del df; gc.collect()
    return pvid, idmap


def load_annot(vid, lab, adir, idmap):
    fp = _find_pq(adir, lab, vid)
    if fp is None:
        return None
    df = pd.read_parquet(fp)
    if not {"agent_id", "target_id", "action", "start_frame", "stop_frame"}.issubset(df.columns):
        return None
    df["agent_id"]  = df["agent_id"].map(lambda x: idmap.get(x, str(x)))
    df["target_id"] = df["target_id"].map(lambda x: idmap.get(x, str(x)))
    return df


# COMPACT DESCRIPTORS (for clustering)

In [4]:
def _ctr(pvid, m):
    pts = list(set(c[1] for c in pvid.columns if c[0] == m))
    for p in ["body_center", "neck", "nose"]:
        if p in pts:
            try:
                return pvid[(m, p, "x")], pvid[(m, p, "y")]
            except Exception:
                continue
    xs = [pvid[(m, p, "x")] for p in pts if (m, p, "x") in pvid.columns]
    ys = [pvid[(m, p, "y")] for p in pts if (m, p, "y") in pvid.columns]
    if xs:
        return pd.concat(xs, axis=1).mean(axis=1), pd.concat(ys, axis=1).mean(axis=1)
    return None, None


def _sp(cx, cy, lag=1):
    return np.sqrt(cx.diff(lag) ** 2 + cy.diff(lag) ** 2)


def _pts(pvid, m):
    return list(set(c[1] for c in pvid.columns if c[0] == m))


# Number of descriptor columns (must match what desc_single / desc_pair produce)
DESC_SINGLE_COLS = 10
DESC_PAIR_COLS   = 9


def desc_single(pvid, m, fps):
    cx, cy = _ctr(pvid, m)
    if cx is None:
        return None
    pts = _pts(pvid, m)
    cols = []
    for lg in [1, 3, 10, 30]:
        cols.append(_sp(cx, cy, _sc(lg, fps)).fillna(0).values)
    cols.append(_sp(cx, cy, 1).fillna(0).diff().fillna(0).values if hasattr(_sp(cx, cy, 1), 'diff')
                else np.gradient(_sp(cx, cy, 1).fillna(0).values))
    # Use a simpler acceleration
    sp1 = _sp(cx, cy, 1).fillna(0).values
    cols[-1] = np.gradient(sp1)  # replace with clean gradient

    if "nose" in pts and "tail_base" in pts:
        bl = np.sqrt(
            (pvid[(m, "nose", "x")] - pvid[(m, "tail_base", "x")]) ** 2
            + (pvid[(m, "nose", "y")] - pvid[(m, "tail_base", "y")]) ** 2
        )
        cols.append(bl.fillna(bl.median()).values)
        cols.append(np.gradient(bl.fillna(bl.median()).values))
        hx = pvid[(m, "nose", "x")] - pvid[(m, "tail_base", "x")]
        hy = pvid[(m, "nose", "y")] - pvid[(m, "tail_base", "y")]
        cols.append(np.gradient(np.arctan2(hy, hx).values))
    else:
        cols.extend([np.zeros(len(pvid))] * 3)

    if "ear_left" in pts and "ear_right" in pts:
        cols.append(
            np.sqrt(
                (pvid[(m, "ear_left", "x")] - pvid[(m, "ear_right", "x")]) ** 2
                + (pvid[(m, "ear_left", "y")] - pvid[(m, "ear_right", "y")]) ** 2
            ).fillna(0).values
        )
    else:
        cols.append(np.zeros(len(pvid)))

    return np.nan_to_num(np.column_stack(cols).astype("float32"))


def desc_pair(pvid, a, b, fps):
    ax, ay = _ctr(pvid, a)
    bx, by = _ctr(pvid, b)
    if ax is None or bx is None:
        return None
    pa, pb = _pts(pvid, a), _pts(pvid, b)
    cols = []
    d = np.sqrt((ax - bx) ** 2 + (ay - by) ** 2)
    cols.append(d.fillna(d.median()).values)
    cols.append(np.gradient(d.fillna(d.median()).values))
    cols.append(d.diff(_sc(10, fps)).fillna(0).values)
    cols.append(_sp(ax, ay).fillna(0).values)
    cols.append(_sp(bx, by).fillna(0).values)
    vax, vay = ax.diff().fillna(0), ay.diff().fillna(0)
    vbx, vby = bx.diff().fillna(0), by.diff().fillna(0)
    dot = vax * vbx + vay * vby
    ma = np.sqrt(vax ** 2 + vay ** 2) + 1e-8
    mb = np.sqrt(vbx ** 2 + vby ** 2) + 1e-8
    cols.append((dot / (ma * mb)).fillna(0).values)
    if "nose" in pa and "tail_base" in pa:
        hx = pvid[(a, "nose", "x")] - pvid[(a, "tail_base", "x")]
        hy = pvid[(a, "nose", "y")] - pvid[(a, "tail_base", "y")]
        dx, dy = bx - ax, by - ay
        cols.append(
            ((hx * dx + hy * dy) / ((np.sqrt(hx ** 2 + hy ** 2) + 1e-8) * (np.sqrt(dx ** 2 + dy ** 2) + 1e-8)))
            .fillna(0).values
        )
    else:
        cols.append(np.zeros(len(pvid)))
    if "nose" in pa and "nose" in pb:
        nn = np.sqrt(
            (pvid[(a, "nose", "x")] - pvid[(b, "nose", "x")]) ** 2
            + (pvid[(a, "nose", "y")] - pvid[(b, "nose", "y")]) ** 2
        )
        cols.append(nn.fillna(nn.median()).values)
    else:
        cols.append(d.fillna(d.median()).values)
    return np.nan_to_num(np.column_stack(cols).astype("float32"))


# CLUSTER ENGINE

In [5]:
class CE:
    def __init__(self):
        self.fitted = False

    def fit(self, ss, sp):
        from sklearn.decomposition import PCA

        self.sc_s = StandardScaler().fit(ss)
        self.sc_p = StandardScaler().fit(sp)
        s1 = self.sc_s.transform(ss)
        p1 = self.sc_p.transform(sp)
        self.pca_s = PCA(n_components=min(ss.shape[1], ss.shape[1])).fit(s1)
        self.pca_p = PCA(n_components=min(sp.shape[1], sp.shape[1])).fit(p1)
        e_s = self.pca_s.transform(s1)
        e_p = self.pca_p.transform(p1)
        del s1, p1; gc.collect()
        self.km_s = MiniBatchKMeans(
            n_clusters=min(N_CLUST_S, len(e_s) // 2 + 1),
            random_state=42, batch_size=4096, n_init=3,
        ).fit(e_s)
        self.km_p = MiniBatchKMeans(
            n_clusters=min(N_CLUST_P, len(e_p) // 2 + 1),
            random_state=42, batch_size=4096, n_init=3,
        ).fit(e_p)
        del e_s, e_p; gc.collect()
        self.fitted = True
        return self

    def tr_s(self, raw):
        return self.km_s.transform(self.pca_s.transform(self.sc_s.transform(raw)))

    def tr_p(self, raw):
        return self.km_p.transform(self.pca_p.transform(self.sc_p.transform(raw)))


def cf_single(pvid, m, fps, ce):
    X = pd.DataFrame(index=pvid.index)
    if not ce.fitted:
        return X
    raw = desc_single(pvid, m, fps)
    if raw is None:
        return X
    d = ce.tr_s(raw)
    del raw
    nc = d.shape[1]
    X["cid"] = d.argmin(axis=1).astype("float32")
    X["cdst"] = d.min(axis=1)
    t3 = np.argsort(d, axis=1)[:, :min(3, nc)]
    for k in range(t3.shape[1]):
        X[f"cd{k}"] = d[np.arange(len(d)), t3[:, k]]
    for w in [30, 90]:
        ws = _sc(w, fps)
        for c in range(min(5, nc)):
            X[f"ch{c}w{w}"] = (
                (X["cid"] == c).astype(float)
                .rolling(ws, min_periods=1, center=True).mean()
            )
    del d, t3
    return X.astype("float32")


def cf_pair(pvid, a, b, fps, ce):
    X = pd.DataFrame(index=pvid.index)
    if not ce.fitted:
        return X
    raw = desc_pair(pvid, a, b, fps)
    if raw is None:
        return X
    d = ce.tr_p(raw)
    del raw
    nc = d.shape[1]
    X["pid"] = d.argmin(axis=1).astype("float32")
    X["pdst"] = d.min(axis=1)
    t3 = np.argsort(d, axis=1)[:, :min(3, nc)]
    for k in range(t3.shape[1]):
        X[f"pd{k}"] = d[np.arange(len(d)), t3[:, k]]
    for w in [30, 90]:
        ws = _sc(w, fps)
        for c in range(min(5, nc)):
            X[f"ph{c}w{w}"] = (
                (X["pid"] == c).astype(float)
                .rolling(ws, min_periods=1, center=True).mean()
            )
    del d, t3
    return X.astype("float32")

# RICH HAND-CRAFTED FEATURES

In [6]:
def f_single(pvid, m, fps):
    X = pd.DataFrame(index=pvid.index)
    cx, cy = _ctr(pvid, m)
    if cx is None:
        return X
    pts = _pts(pvid, m)
    sp1 = _sp(cx, cy, 1)
    for lg in [1, 2, 5, 10, 20, 30, 60]:
        X[f"sp{lg}"] = _sp(cx, cy, _sc(lg, fps))
    X["acc"] = sp1.diff()
    for w in [10, 30, 60, 120]:
        ws = _sc(w, fps)
        r = sp1.rolling(ws, min_periods=1, center=True)
        X[f"sm{w}"] = r.mean()
        X[f"ss{w}"] = r.std().fillna(0)
        X[f"sx{w}"] = r.max()
    for sp in [15, 30, 60]:
        X[f"se{sp}"] = sp1.ewm(span=_sc(sp, fps), min_periods=1).mean()
    if "nose" in pts and "tail_base" in pts:
        nx, ny = pvid[(m, "nose", "x")], pvid[(m, "nose", "y")]
        tx, ty = pvid[(m, "tail_base", "x")], pvid[(m, "tail_base", "y")]
        bl = np.sqrt((nx - tx) ** 2 + (ny - ty) ** 2)
        X["bl"] = bl
        for lg in [10, 20, 40]:
            X[f"bld{lg}"] = bl.diff(_sc(lg, fps))
        X["hd"] = np.arctan2(ny - ty, nx - tx)
        X["hdc"] = X["hd"].diff()
        X["cur"] = X["hdc"].rolling(_sc(5, fps), min_periods=1, center=True).mean()
    if "ear_left" in pts and "ear_right" in pts:
        X["ead"] = np.sqrt(
            (pvid[(m, "ear_left", "x")] - pvid[(m, "ear_right", "x")]) ** 2
            + (pvid[(m, "ear_left", "y")] - pvid[(m, "ear_right", "y")]) ** 2
        )
    for bp in ["ear_left", "ear_right", "tail_base"]:
        if bp in pts:
            bx, by = pvid[(m, bp, "x")], pvid[(m, bp, "y")]
            for lg in [1, 10]:
                X[f"{bp[:3]}s{lg}"] = _sp(bx, by, _sc(lg, fps))
    X["px"], X["py"] = cx, cy
    for h in [15, 30, 60]:
        hs = _sc(h, fps)
        sf = _sp(cx.shift(-hs), cy.shift(-hs), hs).abs()
        sp_ = _sp(cx, cy, hs).abs()
        X[f"asy{h}"] = (sf - sp_) / (sf + sp_ + 1e-6)
    return X.astype("float32")


def f_pair(pvid, a, b, fps):
    X = pd.DataFrame(index=pvid.index)
    ax, ay = _ctr(pvid, a)
    bx, by = _ctr(pvid, b)
    if ax is None or bx is None:
        return X
    pa, pb = _pts(pvid, a), _pts(pvid, b)
    d = np.sqrt((ax - bx) ** 2 + (ay - by) ** 2)
    X["d"] = d
    for lg in [1, 5, 10, 20, 30, 60]:
        X[f"dd{lg}"] = d.diff(_sc(lg, fps))
    for w in [10, 30, 60, 120]:
        ws = _sc(w, fps)
        r = d.rolling(ws, min_periods=1, center=True)
        X[f"dm{w}"] = r.mean()
        X[f"ds{w}"] = r.std().fillna(0)
        X[f"dn{w}"] = r.min()
    for t in [2.0, 4.0, 6.0]:
        ct = (d < t).astype("float32")
        X[f"ct{int(t)}"] = ct
        for w in [10, 30, 60]:
            X[f"ct{int(t)}r{w}"] = ct.rolling(_sc(w, fps), min_periods=1, center=True).mean()
    X["app"] = -d.diff()
    sa, sb = _sp(ax, ay), _sp(bx, by)
    X["sa"], X["sb"], X["sr"] = sa, sb, sa / (sb + 1e-6)
    for lg in [5, 10, 20, 30, 60]:
        sc_ = _sc(lg, fps)
        X[f"sA{lg}"] = _sp(ax, ay, sc_)
        X[f"sB{lg}"] = _sp(bx, by, sc_)
    vax, vay = ax.diff(), ay.diff()
    vbx, vby = bx.diff(), by.diff()
    dot = vax * vbx + vay * vby
    ma = np.sqrt(vax ** 2 + vay ** 2) + 1e-8
    mb = np.sqrt(vbx ** 2 + vby ** 2) + 1e-8
    X["va"] = dot / (ma * mb)
    for o in [10, 20, -10, -20]:
        X[f"va{o}"] = X["va"].shift(-_sc(abs(o), fps) * (1 if o > 0 else -1))
    if "nose" in pa and "tail_base" in pa:
        hx = pvid[(a, "nose", "x")] - pvid[(a, "tail_base", "x")]
        hy = pvid[(a, "nose", "y")] - pvid[(a, "tail_base", "y")]
        dx, dy = bx - ax, by - ay
        X["af"] = (hx * dx + hy * dy) / (
            (np.sqrt(hx ** 2 + hy ** 2) + 1e-8) * (np.sqrt(dx ** 2 + dy ** 2) + 1e-8)
        )
    if "nose" in pb and "tail_base" in pb:
        hx2 = pvid[(b, "nose", "x")] - pvid[(b, "tail_base", "x")]
        hy2 = pvid[(b, "nose", "y")] - pvid[(b, "tail_base", "y")]
        X["bf"] = (hx2 * (ax - bx) + hy2 * (ay - by)) / (
            (np.sqrt(hx2 ** 2 + hy2 ** 2) + 1e-8) * (np.sqrt((ax - bx) ** 2 + (ay - by) ** 2) + 1e-8)
        )
    if "nose" in pa and "tail_base" in pa and "nose" in pb and "tail_base" in pb:
        hax = pvid[(a, "nose", "x")] - pvid[(a, "tail_base", "x")]
        hay = pvid[(a, "nose", "y")] - pvid[(a, "tail_base", "y")]
        hbx = pvid[(b, "nose", "x")] - pvid[(b, "tail_base", "x")]
        hby = pvid[(b, "nose", "y")] - pvid[(b, "tail_base", "y")]
        X["ha"] = (hax * hbx + hay * hby) / (
            (np.sqrt(hax ** 2 + hay ** 2) + 1e-8) * (np.sqrt(hbx ** 2 + hby ** 2) + 1e-8)
        )
    if "nose" in pa and "nose" in pb:
        nn = np.sqrt(
            (pvid[(a, "nose", "x")] - pvid[(b, "nose", "x")]) ** 2
            + (pvid[(a, "nose", "y")] - pvid[(b, "nose", "y")]) ** 2
        )
        X["nn"] = nn
        for lg in [5, 10, 20]:
            X[f"nnd{lg}"] = nn.diff(_sc(lg, fps))
        for w in [10, 30]:
            X[f"nnc{w}"] = (nn < 2).astype(float).rolling(_sc(w, fps), min_periods=1, center=True).mean()
        nnx = pvid[(b, "nose", "x")] - pvid[(a, "nose", "x")]
        nny = pvid[(b, "nose", "y")] - pvid[(a, "nose", "y")]
        nm = np.sqrt(nnx ** 2 + nny ** 2) + 1e-8
        vx = pvid[(a, "nose", "x")].diff() * fps
        vy = pvid[(a, "nose", "y")].diff() * fps
        X["nnr"] = vx * (nnx / nm) + vy * (nny / nm)
        X["nnl"] = vx * (-nny / nm) + vy * (nnx / nm)
    if "nose" in pa and "tail_base" in pb:
        X["nt"] = np.sqrt(
            (pvid[(a, "nose", "x")] - pvid[(b, "tail_base", "x")]) ** 2
            + (pvid[(a, "nose", "y")] - pvid[(b, "tail_base", "y")]) ** 2
        )
    if "nose" in pa:
        X["nb"] = np.sqrt(
            (pvid[(a, "nose", "x")] - bx) ** 2 + (pvid[(a, "nose", "y")] - by) ** 2
        )
    return X.astype("float32")


def f_third(pvid, a, b, mice, fps):
    X = pd.DataFrame(index=pvid.index)
    others = [m for m in mice if m not in (a, b)]
    if not others:
        return X
    ax, ay = _ctr(pvid, a)
    bx, by = _ctr(pvid, b)
    if ax is None or bx is None:
        return X
    dla, dlb = [], []
    for m in others:
        mx, my = _ctr(pvid, m)
        if mx is None:
            continue
        dla.append(np.sqrt((mx - ax) ** 2 + (my - ay) ** 2))
        dlb.append(np.sqrt((mx - bx) ** 2 + (my - by) ** 2))
    if not dla:
        return X
    da = pd.concat(dla, axis=1)
    db = pd.concat(dlb, axis=1)
    X["tda"] = da.min(axis=1)
    X["tdb"] = db.min(axis=1)
    pd_ = np.sqrt((ax - bx) ** 2 + (ay - by) ** 2) + 1e-6
    X["tca"] = (da < 4).sum(axis=1).astype("float32")
    X["tbt"] = (da.values + db.values).min(axis=1) / pd_.values
    return X.astype("float32")


def f_meta(row, aid, tid):
    f = {}
    sm = {"male": 0.0, "female": 1.0}
    for px, ml in [("a", str(aid)), ("t", str(tid))]:
        n = ml.replace("mouse", "").replace("self", "0")
        try:
            n = int(n)
        except Exception:
            n = 0
        f[f"{px}s"] = sm.get(str(row.get(f"mouse{n}_sex", "")).strip().lower(), -1.0)
        f[f"{px}t"] = float(hash(str(row.get(f"mouse{n}_strain", "")).strip().lower()) % 100) / 100.0
    f["ss"] = float(f["as"] == f["ts"])
    f["fp"] = float(row.get("frames_per_second", 30))
    aw = float(row.get("arena_width_cm", 50))
    ah = float(row.get("arena_height_cm", 50))
    f["aa"] = aw * ah
    f["ar"] = aw / (ah + 1e-6)
    f["ash"] = {"square": 0.0, "circular": 1.0, "rectangular": 2.0}.get(
        str(row.get("arena_shape", "")).strip().lower(), -1.0
    )
    return f


# BUILD FULL FEATURES

In [7]:
def build_X(pvid, row, aid, tid, mice, fps, sc, ce):
    nf = len(pvid)
    parts = []
    if aid not in sc:
        sc[aid] = f_single(pvid, aid, fps)
    parts.append(sc[aid].add_prefix("A_"))
    is_self = tid == "self" or tid == aid
    tm = aid if is_self else tid
    if not is_self:
        if tm not in sc:
            sc[tm] = f_single(pvid, tm, fps)
        parts.append(sc[tm].add_prefix("B_"))
        parts.append(f_pair(pvid, aid, tm, fps))
        parts.append(f_third(pvid, aid, tm, mice, fps))
    parts.append(cf_single(pvid, aid, fps, ce).add_prefix("cA_"))
    if not is_self:
        parts.append(cf_single(pvid, tm, fps, ce).add_prefix("cB_"))
        parts.append(cf_pair(pvid, aid, tm, fps, ce).add_prefix("cP_"))
    mf = f_meta(row, aid, tid)
    mf["slf"] = float(is_self)
    mf_df = pd.DataFrame(
        {k: np.full(nf, v, dtype="float32") for k, v in mf.items()}, index=pvid.index
    )
    parts.append(mf_df)
    result = sanitize(pd.concat(parts, axis=1)).astype("float32")
    # Free the individual parts
    del parts
    return result


def frame_labels(annot, nf, aid, tid, act):
    y = np.zeros(nf, dtype="float32")
    if annot is None or annot.empty:
        return y
    tid_actual = aid if tid == "self" else tid
    mask = (annot["agent_id"] == aid) & (annot["target_id"] == tid_actual) & (annot["action"] == act)
    for _, r in annot[mask].iterrows():
        s = max(0, int(r["start_frame"]))
        e = min(int(r["stop_frame"]) + 1, nf)
        y[s:e] = 1.0
    if y.sum() == 0 and tid == "self":
        mask2 = (annot["agent_id"] == aid) & (annot["target_id"] == "self") & (annot["action"] == act)
        for _, r in annot[mask2].iterrows():
            s = max(0, int(r["start_frame"]))
            e = min(int(r["stop_frame"]) + 1, nf)
            y[s:e] = 1.0
    return y


def sample_idx(y, nf, pad=PAD_FRAMES, neg_ratio=NEG_RATIO, max_t=MAX_FRAMES_VID):
    pi = np.where(y == 1)[0]
    if len(pi) == 0:
        return np.random.RandomState(42).choice(nf, min(300, nf), replace=False)
    keep = set()
    for i in pi:
        for j in range(max(0, i - pad), min(nf, i + pad + 1)):
            keep.add(j)
    keep = np.array(sorted(keep))
    neg = np.setdiff1d(np.arange(nf), keep)
    nn = min(len(neg), int(len(pi) * neg_ratio), max_t - len(keep))
    if nn > 0:
        keep = np.sort(np.concatenate([keep, np.random.RandomState(42).choice(neg, nn, replace=False)]))
    return keep[:max_t]


#  PASS 1: STREAMING SNIPPETS  (reservoir-sampled)

def pass1_snippets(train_df, root):
    print("═" * 60)
    print("  PASS 1: Collecting snippets (reservoir-sampled)")
    print("═" * 60)
    t0 = time.time()
    tdir = root / "train_tracking"
    rng = np.random.RandomState(42)
    indices = rng.permutation(len(train_df))[:UNSUP_MAX_VIDS]

    # We need to know ncols — derive from first successful video
    ncols_s, ncols_p = None, None
    res_s, res_p = None, None

    for cnt, idx in enumerate(indices):
        row = train_df.iloc[idx]
        pvid, _ = load_track(
            row["video_id"], row["lab_id"], tdir, float(row.get("pix_per_cm_approx", 10))
        )
        if pvid is None:
            continue
        fps = float(row.get("frames_per_second", 30))
        mice = sorted(set(c[0] for c in pvid.columns))

        for m in mice:
            raw = desc_single(pvid, m, fps)
            if raw is not None:
                if ncols_s is None:
                    ncols_s = raw.shape[1]
                    res_s = Reservoir(MAX_RESERVOIR_S, ncols_s)
                if len(raw) > SNIPPET_SAMPLE:
                    raw = raw[rng.choice(len(raw), SNIPPET_SAMPLE, replace=False)]
                res_s.add(raw)
                del raw

        for i, m1 in enumerate(mice):
            for m2 in mice[i + 1:]:
                raw = desc_pair(pvid, m1, m2, fps)
                if raw is not None:
                    if ncols_p is None:
                        ncols_p = raw.shape[1]
                        res_p = Reservoir(MAX_RESERVOIR_P, ncols_p)
                    if len(raw) > SNIPPET_SAMPLE:
                        raw = raw[rng.choice(len(raw), SNIPPET_SAMPLE, replace=False)]
                    res_p.add(raw)
                    del raw

        del pvid
        if (cnt + 1) % 500 == 0:
            s_n = res_s.filled if res_s else 0
            p_n = res_p.filled if res_p else 0
            print(f"    {cnt+1}/{len(indices)}  s={s_n:,}  p={p_n:,}  ({time.time()-t0:.0f}s)")

    if res_s is None or res_p is None:
        sys.exit("[FATAL] No snippets collected")

    S = res_s.get()
    P = res_p.get()
    del res_s, res_p
    gc.collect()
    print(f"  Done: S={S.shape}  P={P.shape}  ({time.time()-t0:.0f}s)")
    return S, P


#  PASS 2: BUILD LABELED DATA → FLUSH CHUNKS TO DISK

def pass2_labeled(train_df, root, ce):
    """
    Extract features per video and flush to disk as .npz chunks.
    Returns: dict[action] -> list of chunk file paths + metadata
    """
    print("═" * 60)
    print("  PASS 2: Building labeled features (disk-backed)")
    print("═" * 60)
    t0 = time.time()
    tdir = root / "train_tracking"
    adir = root / "train_annotation"
    tmpdir = get_tmpdir()

    # Track: action -> list of (chunk_path, n_pos, n_total)
    chunk_registry = defaultdict(list)
    # Track column names per action (set on first chunk)
    col_registry = {}

    nok, nskip, nnob, npairs = 0, 0, 0, 0

    for _, row in train_df.iterrows():
        vid, lab = row["video_id"], row["lab_id"]
        fps = float(row.get("frames_per_second", 30))
        ppc = float(row.get("pix_per_cm_approx", 10))
        behaviors = parse_beh(row["behaviors_labeled"])
        if not behaviors:
            nnob += 1
            continue
        if _find_pq(adir, lab, vid) is None:
            nskip += 1
            continue
        pvid, idmap = load_track(vid, lab, tdir, ppc)
        if pvid is None:
            nskip += 1
            continue
        nok += 1
        try:
            annot = load_annot(vid, lab, adir, idmap)
            nf = len(pvid)
            mice = sorted(set(c[0] for c in pvid.columns))
            sc = {}

            if nok <= 3:
                am = annot["agent_id"].unique().tolist()[:6] if annot is not None else []
                print(f"    DEBUG vid={vid}: mice={mice}  annot_agents={am}")

            for aid, tid, act in behaviors:
                if aid not in mice:
                    continue
                tm = aid if tid == "self" else tid
                if tid != "self" and tm not in mice:
                    continue

                yf = frame_labels(annot, nf, aid, tid, act)
                ki = sample_idx(yf, nf)
                y = yf[ki]
                X = build_X(pvid, row, aid, tid, mice, fps, sc, ce).iloc[ki]

                # Record column names on first encounter
                if act not in col_registry:
                    col_registry[act] = list(X.columns)

                # Ensure consistent columns
                X = X.reindex(columns=col_registry[act], fill_value=0.0)

                # Flush to disk as compressed npz
                chunk_path = tmpdir / f"{act}_{nok}_{aid}_{tid}.npz"
                np.savez_compressed(
                    chunk_path,
                    X=X.values.astype("float32"),
                    y=y.astype("float32"),
                    vid=np.array([vid] * len(ki), dtype="int64"),
                )
                chunk_registry[act].append((str(chunk_path), int(y.sum()), len(y)))
                npairs += 1

                del X, y, ki, yf
            del sc
        except Exception as e:
            print(f"  [ERR] {vid}: {e}")
            traceback.print_exc()

        del pvid
        if nok % 30 == 0:
            gc.collect()
        if nok % 100 == 0 or nok <= 3:
            print(
                f"  [{nok:>4}ok/{nskip:>4}skip/{nnob:>5}nob] pairs={npairs} "
                f"acts={len(chunk_registry)} ({time.time()-t0:.0f}s)"
            )

    print(f"\n  Done: {nok} vids, {npairs} pairs, {len(chunk_registry)} actions ({time.time()-t0:.0f}s)")
    for act in sorted(chunk_registry.keys()):
        ny = sum(c[1] for c in chunk_registry[act])
        nt = sum(c[2] for c in chunk_registry[act])
        print(f"    {act:25s}  pos={ny:>8,}  total={nt:>10,}")

    return chunk_registry, col_registry

# TRAIN XGBOOST — STREAMING FROM DISK CHUNKS

In [8]:
def _load_and_cap_chunks(chunks, max_rows, rng):
    """
    Load chunks from disk and cap total rows.
    Returns X (np.float32), y (np.float32), g (np.int64).
    """
    # First pass: compute total size
    total = sum(c[2] for c in chunks)

    if total <= max_rows:
        # Load all
        Xs, ys, gs = [], [], []
        for path, _, _ in chunks:
            d = np.load(path)
            Xs.append(d["X"])
            ys.append(d["y"])
            gs.append(d["vid"])
            del d
        return np.vstack(Xs), np.concatenate(ys), np.concatenate(gs)

    # Need to subsample: load chunk by chunk, reservoir sample
    keep_ratio = max_rows / total
    Xs, ys, gs = [], [], []
    cur = 0
    for path, npos, ntot in chunks:
        d = np.load(path)
        X_c, y_c, g_c = d["X"], d["y"], d["vid"]
        del d
        # Always keep positives, subsample negatives
        pos_mask = y_c == 1
        neg_mask = ~pos_mask
        neg_keep = int(neg_mask.sum() * keep_ratio)
        if neg_keep < neg_mask.sum() and neg_mask.sum() > 0:
            neg_idx = rng.choice(np.where(neg_mask)[0], neg_keep, replace=False)
            pos_idx = np.where(pos_mask)[0]
            idx = np.sort(np.concatenate([pos_idx, neg_idx]))
            X_c, y_c, g_c = X_c[idx], y_c[idx], g_c[idx]
        Xs.append(X_c)
        ys.append(y_c)
        gs.append(g_c)
        cur += len(X_c)
        del X_c, y_c, g_c
        if cur >= max_rows:
            break
    return np.vstack(Xs), np.concatenate(ys), np.concatenate(gs)


def train_xgb(chunk_registry, col_registry):
    from xgboost import XGBClassifier

    models = {}
    rng = np.random.RandomState(42)

    for act in sorted(chunk_registry.keys()):
        chunks = chunk_registry[act]
        if not chunks:
            continue
        fcols = col_registry[act]
        total_pos = sum(c[1] for c in chunks)
        total_n = sum(c[2] for c in chunks)
        print(f"\n{'─'*55}\n  {act:25s}  pos={total_pos:>8,}  total={total_n:>10,}  feats={len(fcols)}")
        if total_pos < 5:
            print("  → skip")
            continue

        # Load with cap
        X, y, g = _load_and_cap_chunks(chunks, MAX_ROWS_PER_ACT, rng)
        np_ = int(y.sum())
        nn_ = len(y) - np_
        print(f"  Loaded: pos={np_:,}  neg={nn_:,}  shape={X.shape}")

        # Further cap negatives
        mx = min(nn_, max(np_ * 15, 150_000))
        if nn_ > mx:
            pi = np.where(y == 1)[0]
            ni = rng.choice(np.where(y == 0)[0], mx, replace=False)
            k = np.sort(np.concatenate([pi, ni]))
            X, y, g = X[k], y[k], g[k]

        spw = max(1.0, min(nn_ / (np_ + 1), 50))
        mdl = XGBClassifier(
            n_estimators=400, max_depth=7, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.6, scale_pos_weight=spw,
            tree_method="hist", random_state=42, n_jobs=-1,
            eval_metric="logloss", early_stopping_rounds=30,
        )
        ug = np.unique(g)
        rng.shuffle(ug)
        sp = max(1, int(len(ug) * 0.8))
        ts = set(ug[:sp])
        tm = np.array([gi in ts for gi in g])
        Xt, yt = X[tm], y[tm]
        Xv, yv = X[~tm], y[~tm]

        if len(Xv) == 0 or yv.sum() == 0:
            mdl.set_params(early_stopping_rounds=None)
            mdl.fit(Xt, yt)
        else:
            mdl.fit(Xt, yt, eval_set=[(Xv, yv)], verbose=False)

        thr = 0.27
        if len(Xv) > 0 and yv.sum() > 0:
            pr = mdl.predict_proba(Xv)[:, 1]
            bf, bt = 0, 0.27
            for t in np.arange(0.05, 0.85, 0.02):
                p = (pr >= t).astype(int)
                tp = ((p == 1) & (yv == 1)).sum()
                fp = ((p == 1) & (yv == 0)).sum()
                fn = ((p == 0) & (yv == 1)).sum()
                prec = tp / (tp + fp + 1e-8)
                rec = tp / (tp + fn + 1e-8)
                f1 = 2 * prec * rec / (prec + rec + 1e-8)
                if f1 > bf:
                    bf, bt = f1, t
            thr = bt
            print(f"  thr={thr:.2f}  F1={bf:.4f}")

        models[act] = (mdl, fcols, thr)
        joblib.dump((mdl, fcols, thr), MDIR / f"m_{act}.joblib")
        print(f"  → saved ({len(fcols)} feats)")
        del X, y, g, Xt, yt, Xv, yv
        gc.collect()

    return models

#  INFERENCE

In [9]:
def to_segs(pr, thr, gap=3, ml=1):
    sm = uniform_filter1d(pr.astype("float64"), size=7)
    bi = (sm >= thr).astype(int)
    segs, ins, st = [], False, 0
    for i in range(len(bi)):
        if bi[i] == 1 and not ins:
            st, ins = i, True
        elif bi[i] == 0 and ins:
            segs.append((st, i - 1))
            ins = False
    if ins:
        segs.append((st, len(bi) - 1))
    mg = []
    for s in segs:
        if mg and s[0] - mg[-1][1] <= gap:
            mg[-1] = (mg[-1][0], s[1])
        else:
            mg.append(s)
    return [(a, b) for a, b in mg if b - a + 1 >= ml]


def predict_test(test_df, root, models, ce):
    print("═" * 60)
    print("  INFERENCE")
    print("═" * 60)
    tdir = root / "test_tracking"
    rows = []
    for _, row in test_df.iterrows():
        vid, lab = row["video_id"], row["lab_id"]
        fps = float(row.get("frames_per_second", 30))
        ppc = float(row.get("pix_per_cm_approx", 10))
        pvid, _ = load_track(vid, lab, tdir, ppc)
        if pvid is None:
            print(f"  [WARN] no tracking {vid}")
            continue
        behaviors = parse_beh(row["behaviors_labeled"])
        mice = sorted(set(c[0] for c in pvid.columns))
        print(f"  vid={vid} frames={len(pvid)} mice={mice} behs={len(behaviors)}")
        sc = {}
        np_ = 0
        for aid, tid, act in behaviors:
            if act not in models:
                continue
            mdl, fcols, thr = models[act]
            if aid not in mice:
                continue
            tm = aid if tid == "self" else tid
            if tid != "self" and tm not in mice:
                continue
            try:
                X = build_X(pvid, row, aid, tid, mice, fps, sc, ce)
                X = X.reindex(columns=fcols, fill_value=0.0).astype("float32")
                pr = mdl.predict_proba(X.values)[:, 1]
                for s, e in to_segs(pr, thr):
                    rows.append({
                        "video_id": vid, "agent_id": aid, "target_id": tid,
                        "action": act, "start_frame": s, "stop_frame": e,
                    })
                np_ += 1
                del X, pr
            except Exception as ex:
                print(f"    [ERR] {aid},{tid},{act}: {ex}")
        print(f"    → {np_} pairs, {len(rows)} segments")
        del pvid, sc
        gc.collect()
    return row

#  Main For Running

In [10]:
def main():
    T0 = time.time()
    print("=" * 60)
    print("  MABe v7-LOWMEM — Memory-Optimized 2-Pass Pipeline")
    print("=" * 60)
    root = find_root()
    for nm in ["train_tracking", "train_annotation", "test_tracking"]:
        d = root / nm
        if d.exists():
            print(f"  {nm:20s} → {sum(1 for _ in d.rglob('*.parquet')):>5} parquets")
        else:
            sys.exit(f"  {nm} MISSING!")
    train_df = pd.read_csv(root / "train.csv")
    test_df = pd.read_csv(root / "test.csv")
    adir = root / "train_annotation"
    n_lab = sum(
        1 for _, r in train_df.iterrows()
        if parse_beh(r["behaviors_labeled"]) and _find_pq(adir, r["lab_id"], r["video_id"])
    )
    n_unlab = len(train_df) - n_lab
    print(f"  {len(train_df)} vids: {n_lab} labeled, {n_unlab} unlabeled, {len(test_df)} test")

    # ── Pass 1: snippets (reservoir-sampled, bounded memory) ──
    snip_s, snip_p = pass1_snippets(train_df, root)

    # ── Fit clustering ──
    print(f"\n  Fitting clusters:  single={N_CLUST_S}  pair={N_CLUST_P}")
    t1 = time.time()
    ce = CE().fit(snip_s, snip_p)
    del snip_s, snip_p
    gc.collect()
    print(f"  Done in {time.time()-t1:.0f}s")

    # ── Pass 2: labeled features → disk chunks ──
    chunk_registry, col_registry = pass2_labeled(train_df, root, ce)
    if not chunk_registry:
        print("[FATAL] No actions!")
        cleanup_tmpdir()
        sys.exit(1)

    # ── Train (streaming from disk) ──
    print(f"\n{'═'*60}\n  TRAINING\n{'═'*60}")
    models = train_xgb(chunk_registry, col_registry)
    del chunk_registry, col_registry
    gc.collect()
    print(f"\n  {len(models)} models: {sorted(models.keys())}")

    # ── Clean up temp chunks ──
    cleanup_tmpdir()

    # ── Predict ──
    rows = predict_test(test_df, root, models, ce)

    # ── Submission ──
    if rows:
        sub = pd.DataFrame(rows)
        sub.insert(0, "row_id", range(len(sub)))
        sub = sub[["row_id", "video_id", "agent_id", "target_id", "action", "start_frame", "stop_frame"]]
    else:
        sub = pd.DataFrame(columns=[
            "row_id", "video_id", "agent_id", "target_id",
            "action", "start_frame", "stop_frame",
        ])
    out = OUT / "submission.csv"
    sub.to_csv(out, index=False)
    print(f"\n  submission.csv → {out}  ({len(sub)} rows)")
    if len(sub):
        print(sub.head(20).to_string())
    print(f"\n  Total time: {time.time()-T0:.0f}s")


if __name__ == "__main__":
    main()

  MABe v7-LOWMEM — Memory-Optimized 2-Pass Pipeline
  train_tracking       →  8742 parquets
  train_annotation     →   863 parquets
  test_tracking        →     1 parquets
  8790 vids: 863 labeled, 7927 unlabeled, 1 test
════════════════════════════════════════════════════════════
  PASS 1: Collecting snippets (reservoir-sampled)
════════════════════════════════════════════════════════════
    500/1200  s=115,440  p=111,120  (599s)
    1000/1200  s=150,000  p=150,000  (1149s)
  Done: S=(150000, 9)  P=(150000, 8)  (1370s)

  Fitting clusters:  single=50  pair=80
  Done in 2s
════════════════════════════════════════════════════════════
  PASS 2: Building labeled features (disk-backed)
════════════════════════════════════════════════════════════
    DEBUG vid=209576908: mice=['mouse1', 'mouse2', 'mouse3', 'mouse4']  annot_agents=['mouse1', 'mouse2', 'mouse4', 'mouse3']
  [   1ok/   2skip/    0nob] pairs=76 acts=7 (64s)
    DEBUG vid=278643799: mice=['mouse1', 'mouse2', 'mouse3', 'mouse4']

ValueError: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().